# CNN-Transformer Smart Insole — 方案A 双输出头 (Dual Output Head v3)
## Leave-One-Subject-Out (LOSO) Cross-Validation

**Architecture:** CNN spatial encoder → Positional Encoding → Transformer Encoder → mean pooling → **Dual FC Heads**

---

## 方案A — 双输出头设计说明

### 问题背景

v2/v3 实验结果显示，heel 和 toe 的 MAE 比 hip/knee 高出 10–40 倍：

| 关节组 | 平均 MAE (v3) | 特点 |
|--------|-------------|------|
| hip, knee（近端） | ~0.01–0.05 | 运动平滑，幅度小，易预测 |
| heel, toe（远端） | ~0.08–0.15 | 运动非线性，幅度大，难预测 |

### 解决方案：拆分输出头

原来 v3 的单个 FC 头：`Linear(128 → 24)`  
所有 8 个关节共用同一组参数，近端关节「抢占」了本应给远端关节的容量。

本文件改为双输出头：

| 组件 | v3（单头） | v3-DualHead（本文件）|
|------|-----------|--------------------|
| 输出层 | 单个 FC Linear (24-dim) | proximal head (12-dim, 1层) + distal head (12-dim, 2层 MLP) |
| Distal 参数容量 | 与 proximal 共享 | 专属 hidden layer，专注 heel/toe 非线性 |
| 损失函数 | 加权 SmoothL1 | `L_proximal + 2×L_distal`（distal 权重加倍）|

### 架构对比

```
[v3 单头]
Transformer feature (B, 128)
  └── Linear(128 → 24) → 8 joints × xyz

[本文件 双头]
Transformer feature (B, 128)
  ├── head_proximal: Linear(128 → 12)                   → hip×2, knee×2
  └── head_distal:  Linear(128 → 64) → ReLU → Linear(64 → 12) → heel×2, toe×2
```

### 为什么 distal head 需要额外一层？

Heel/toe 的轨迹在 heel-strike 和 toe-off 瞬间有剧烈非线性跳变。  
单层 Linear 只能学线性映射，无法捕捉这种非线性。  
增加一个隐层（Linear → ReLU → Linear）让 distal head 具备学习非线性特征的能力，  
而不影响近端关节（hip/knee）的训练。

> **Note:** Subjects 1 和 3 因电池引起的时间戳漂移被排除。LOSO 在剩余 3 个 subjects 上运行。

## 1. Imports and paths

In [ ]:
from pathlib import Path
import random
import math
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

BASE       = Path('data')
SYNCED_DIR = BASE / 'synced'
MODELS_DIR = BASE / 'models_transformer_dualhead'   # dual-head v3
MODELS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Device:', DEVICE)
print('Synced dir:', SYNCED_DIR)
print('Models dir:', MODELS_DIR)

## 2. Joint setup and dual-head split

关键设计：OUTPUT_JOINTS 的顺序必须是「近端关节在前，远端关节在后」。  
这样将 Y 展平为 (B, 24) 后，前 12 维自然对应 proximal head，后 12 维对应 distal head，  
无需任何额外的索引操作。

In [ ]:
TARGET_JOINTS = [
    'left-hip',
    'right-hip',
    'left-knee',
    'right-knee',
    'left-heel',
    'right-heel',
    'left-big-toe-tip',
    'right-big-toe-tip',
]
OUTPUT_JOINTS = TARGET_JOINTS.copy()

CONNECTIONS = [
    ('left-hip',   'left-knee'),
    ('left-knee',  'left-heel'),
    ('left-heel',  'left-big-toe-tip'),
    ('right-hip',  'right-knee'),
    ('right-knee', 'right-heel'),
    ('right-heel', 'right-big-toe-tip'),
    ('left-hip',   'right-hip'),
]

# ── 双输出头关节分组 ─────────────────────────────────────────
# OUTPUT_JOINTS 顺序：proximal (indices 0-3) → distal (indices 4-7)
# Y 展平后：Y[:, :12] = proximal, Y[:, 12:] = distal
PROXIMAL_JOINTS = ['left-hip', 'right-hip', 'left-knee', 'right-knee']
DISTAL_JOINTS   = ['left-heel', 'right-heel', 'left-big-toe-tip', 'right-big-toe-tip']

PROXIMAL_DIM = len(PROXIMAL_JOINTS) * 3   # 12  (4 joints × xyz)
DISTAL_DIM   = len(DISTAL_JOINTS)   * 3   # 12  (4 joints × xyz)

assert PROXIMAL_JOINTS + DISTAL_JOINTS == OUTPUT_JOINTS, \
    'PROXIMAL + DISTAL must cover OUTPUT_JOINTS in the same order'

print(f'Proximal joints ({len(PROXIMAL_JOINTS)}): {PROXIMAL_JOINTS}')
print(f'Distal   joints ({len(DISTAL_JOINTS)}):   {DISTAL_JOINTS}')
print(f'PROXIMAL_DIM={PROXIMAL_DIM}, DISTAL_DIM={DISTAL_DIM}')

def as_str_list(names):
    return [n.decode('utf-8') if isinstance(n, bytes) else str(n) for n in names]

## 3. Load synced sessions

In [ ]:
def load_synced_sessions(synced_dir):
    session_files = sorted(synced_dir.glob('*_synced.npz'))
    if not session_files:
        raise FileNotFoundError(f'No synced files found in {synced_dir}')
    sessions = []
    for path in session_files:
        data = np.load(path, allow_pickle=True)
        insoles     = np.asarray(data['insoles'],     dtype=np.float32)
        joints      = np.asarray(data['joints'],      dtype=np.float32)
        joint_names = as_str_list(data['joint_names'])
        timestamps  = np.asarray(data['timestamps'],  dtype=np.float64)
        sessions.append({
            'name':        path.stem.replace('_synced', ''),
            'insoles':     insoles,
            'joints':      joints,
            'joint_names': joint_names,
            'timestamps':  timestamps,
        })
        print(f'{path.name}: insoles={insoles.shape}, joints={joints.shape}')
    return sessions

sessions = load_synced_sessions(SYNCED_DIR)
print(f'\nTotal subjects loaded: {len(sessions)}')
assert len(sessions) >= 2, f'Need at least 2 subjects for LOSO, found {len(sessions)}'

## 4. Session preprocessing

In [ ]:
def replace_nan_with_zero(x):
    return np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def build_joint_index(joint_names):
    mapping = {name: i for i, name in enumerate(joint_names)}
    missing = [name for name in TARGET_JOINTS if name not in mapping]
    if missing:
        raise ValueError(f'Missing required joints: {missing}')
    return mapping

def select_output_targets(joints, joint_names):
    name_to_idx  = build_joint_index(joint_names)
    keep_indices = [name_to_idx[name] for name in OUTPUT_JOINTS]
    return joints[:, keep_indices, :].astype(np.float32)

def preprocess_sessions(sessions):
    processed = []
    for session in sessions:
        x = replace_nan_with_zero(session['insoles'])
        y = select_output_targets(session['joints'], session['joint_names'])
        processed.append({
            'name':       session['name'],
            'X':          x,
            'Y':          y,
            'Y_raw':      y.copy(),
            'timestamps': session['timestamps'],
        })
        print(f"{session['name']}: X={x.shape}, Y={y.shape}")
    return processed

processed_sessions = preprocess_sessions(sessions)

## 5. Model configuration

In [ ]:
SEQ_LEN     = 60   # [v3] 30→60: covers one full gait cycle (~2 s at 30 fps)
SEQ_STRIDE  = 1
BATCH_SIZE  = 32
EPOCHS      = 400
PATIENCE    = 20
VAL_RATIO   = 0.15

# CNN encoder
CNN_CHANNELS = (16, 32)

# Transformer
D_MODEL                = 128
NHEAD                  = 4
NUM_TRANSFORMER_LAYERS = 2
DIM_FEEDFORWARD        = 256
DROPOUT                = 0.2

print('Config OK')
print(f'  SEQ_LEN={SEQ_LEN}, BATCH_SIZE={BATCH_SIZE}, EPOCHS={EPOCHS}, PATIENCE={PATIENCE}')
print(f'  D_MODEL={D_MODEL}, NHEAD={NHEAD}, LAYERS={NUM_TRANSFORMER_LAYERS}, FFN={DIM_FEEDFORWARD}')
print(f'  DROPOUT={DROPOUT}')

## 6. Build temporal sequences

与 v3 完全相同：输入为 (T, 4, H, W)，包含原始压力和速度差分通道。

In [ ]:
class SlidingWindowDataset(Dataset):
    def __init__(self, sessions, items, seq_len):
        self.sessions = sessions
        self.items    = items
        self.seq_len  = seq_len

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        session_idx, end_idx = self.items[idx]
        s         = self.sessions[session_idx]
        start_idx = end_idx - self.seq_len + 1

        x = s['X'][start_idx : end_idx + 1]          # (T, 2, H, W)
        x_diff = np.diff(x, axis=0, prepend=x[:1])   # (T, 2, H, W) velocity diff
        x_input = np.concatenate([x, x_diff], axis=1).astype(np.float32)  # (T, 4, H, W)

        y     = s['Y'][end_idx]
        y_raw = s['Y_raw'][end_idx]
        return {
            'X':       torch.from_numpy(x_input).float(),
            'Y':       torch.from_numpy(y).float(),
            'Y_raw':   torch.from_numpy(y_raw).float(),
            'session': s['name'],
        }


def build_items(sessions_subset, seq_len):
    items = []
    for local_i, s in enumerate(sessions_subset):
        T = len(s['X'])
        if T < seq_len:
            print(f"  WARNING: {s['name']} has only {T} frames < SEQ_LEN={seq_len}, skipped.")
            continue
        for end_idx in range(seq_len - 1, T):
            items.append((local_i, end_idx))
    return items

print('Dataset helpers defined.')
print('Input channels per frame: 4  (2 pressure + 2 velocity diff)')

## 7. CNN-Transformer Dual-Head Model

### 完整架构图

```
Input  (B, T, 4, H, W)   ← 4 channels: 2 pressure + 2 velocity diff
  │
  ├─ CNN FrameEncoder (per frame)  ──► (B, T, 32)   spatial features
  ├─ Linear projection             ──► (B, T, D_MODEL=128)
  ├─ Sinusoidal Positional Encoding
  ├─ Transformer Encoder (2 layers, nhead=4)
  │       └─ Self-attention across T=60 timesteps (~2 s)
  └─ Mean pooling over T           ──► (B, 128)     shared feature
                │
                ├── head_proximal: Linear(128 → 12)
                │       └─ hip_L(3) + hip_R(3) + knee_L(3) + knee_R(3)
                │
                └── head_distal: Linear(128 → 64) → ReLU → Dropout → Linear(64 → 12)
                        └─ heel_L(3) + heel_R(3) + toe_L(3) + toe_R(3)
```

### 双头设计原理

**head_proximal（单层）：** hip 和 knee 的运动较平滑，单层线性映射足够捕捉其规律。  
**head_distal（两层）：** heel/toe 在 heel-strike 和 toe-off 时有剧烈非线性变化，  
需要隐层引入非线性激活函数（ReLU）才能拟合。  
两个头完全独立，参数互不干扰，避免「难关节拉高损失导致对易关节过拟合」的问题。

In [ ]:
# ── Sinusoidal positional encoding ──────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


# ── CNN spatial encoder ──────────────────────────────────────
class FrameEncoder(nn.Module):
    def __init__(self, in_channels=4, c1=16, c2=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, c1, kernel_size=3, padding=1),
            nn.BatchNorm2d(c1), nn.ReLU(),
            nn.Conv2d(c1, c1, kernel_size=3, padding=1),
            nn.BatchNorm2d(c1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(c1, c2, kernel_size=3, padding=1),
            nn.BatchNorm2d(c2), nn.ReLU(),
            nn.Conv2d(c2, c2, kernel_size=3, padding=1),
            nn.BatchNorm2d(c2), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )

    def forward(self, x):
        return torch.flatten(self.encoder(x), start_dim=1)


# ── CNN-Transformer with Dual Output Heads ──────────────────
class CNNTransformerDualHead(nn.Module):
    '''
    方案A — 双输出头架构。

    共享 Transformer encoder 提取全局时序特征后，分叉为两个独立输出头：
      head_proximal: 单层 Linear，预测 hip + knee (近端关节)
      head_distal:   两层 MLP，预测 heel + toe  (远端关节，需要非线性)

    输出: (pred_proximal, pred_distal)
      pred_proximal: (B, PROXIMAL_DIM=12)
      pred_distal:   (B, DISTAL_DIM=12)
    '''
    def __init__(
        self,
        proximal_dim,
        distal_dim,
        in_channels=4,
        c1=16,
        c2=32,
        d_model=128,
        nhead=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.2,
    ):
        super().__init__()

        # ── Shared encoder ───────────────────────────────────
        self.cnn = FrameEncoder(in_channels=in_channels, c1=c1, c2=c2)
        with torch.no_grad():
            dummy       = torch.zeros(1, in_channels, 66, 30)
            encoder_dim = self.cnn(dummy).shape[1]
        print(f'CNN encoder_dim = {encoder_dim}  (in_channels={in_channels})')

        self.input_proj = nn.Linear(encoder_dim, d_model)
        self.pos_enc    = PositionalEncoding(d_model, dropout=dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm        = nn.LayerNorm(d_model)
        self.dropout     = nn.Dropout(dropout)

        # ── Proximal head: hip + knee ─────────────────────────
        # Single Linear — smooth joints, linear mapping is sufficient.
        self.head_proximal = nn.Linear(d_model, proximal_dim)

        # ── Distal head: heel + toe ───────────────────────────
        # Two-layer MLP: hidden layer provides non-linear capacity
        # needed for the abrupt trajectory changes at heel-strike / toe-off.
        self.head_distal = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, distal_dim),
        )

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        x = self.cnn(x)
        x = x.view(B, T, -1)
        x = self.input_proj(x)
        x = self.pos_enc(x)
        x = self.transformer(x)
        x = self.norm(x)
        feat = self.dropout(x.mean(dim=1))   # (B, d_model)

        pred_proximal = self.head_proximal(feat)   # (B, 12)
        pred_distal   = self.head_distal(feat)     # (B, 12)
        return pred_proximal, pred_distal


# ── Factory helpers ──────────────────────────────────────────
def make_model():
    m = CNNTransformerDualHead(
        proximal_dim    = PROXIMAL_DIM,
        distal_dim      = DISTAL_DIM,
        in_channels     = 4,
        c1              = CNN_CHANNELS[0],
        c2              = CNN_CHANNELS[1],
        d_model         = D_MODEL,
        nhead           = NHEAD,
        num_layers      = NUM_TRANSFORMER_LAYERS,
        dim_feedforward = DIM_FEEDFORWARD,
        dropout         = DROPOUT,
    ).to(DEVICE)
    return m


def make_optimizer(model):
    return torch.optim.Adam([
        {'params': model.cnn.parameters(),           'lr': 2e-3},
        {'params': model.input_proj.parameters(),    'lr': 1e-3},
        {'params': model.transformer.parameters(),   'lr': 5e-4},
        {'params': model.head_proximal.parameters(), 'lr': 1e-3},
        {'params': model.head_distal.parameters(),   'lr': 1e-3},
    ], weight_decay=1e-4)


_m = make_model()
n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')
del _m

## 8. Loss function and training helpers

### 双头损失设计

```
L_total = L_proximal + DISTAL_LOSS_WEIGHT × L_distal
```

- `L_proximal`: SmoothL1 loss on hip + knee predictions vs. targets
- `L_distal`:   SmoothL1 loss on heel + toe predictions vs. targets  
- `DISTAL_LOSS_WEIGHT = 2.0`：为 distal head 分配 2 倍梯度权重

**为什么用 2.0 而不是更大的值？**  
v3 的 joint weights 已经试过 `heel/toe = 3.0`，但效果不显著。  
过大的权重会让 proximal head 梯度过小（被忽略），反而破坏近端关节精度。  
`2.0` 是一个较保守的初始值，可在 v3 结果出来后调整。

In [ ]:
DISTAL_LOSS_WEIGHT = 2.0   # heel/toe loss weighted 2× relative to hip/knee


def dual_head_loss(pred_proximal, pred_distal, target_flat):
    '''
    Dual-head weighted SmoothL1 loss.

    target_flat: (B, 24) — OUTPUT_JOINTS order, proximal first
      target_flat[:, :12] → hip×2, knee×2  (proximal)
      target_flat[:, 12:] → heel×2, toe×2  (distal)

    Returns: scalar loss
    '''
    tgt_p = target_flat[:, :PROXIMAL_DIM]   # (B, 12)
    tgt_d = target_flat[:, PROXIMAL_DIM:]   # (B, 12)

    loss_p = F.smooth_l1_loss(pred_proximal, tgt_p)
    loss_d = F.smooth_l1_loss(pred_distal,   tgt_d)
    return loss_p + DISTAL_LOSS_WEIGHT * loss_d


def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, n_samples = 0.0, 0
    with torch.set_grad_enabled(is_train):
        for batch in loader:
            x = batch['X'].to(DEVICE)
            y = batch['Y'].to(DEVICE).view(x.size(0), -1)   # (B, 24)

            pred_p, pred_d = model(x)   # dual-head output
            loss = dual_head_loss(pred_p, pred_d, y)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            bs = x.size(0)
            total_loss += loss.item() * bs
            n_samples  += bs
    return total_loss / max(n_samples, 1)


def evaluate(model, loader, y_mean=None, y_std=None):
    model.eval()
    total_loss, n_samples = 0.0, 0
    preds_list, trues_list = [], []

    with torch.no_grad():
        for batch in loader:
            x     = batch['X'].to(DEVICE)
            y     = batch['Y'].to(DEVICE).view(x.size(0), -1)
            y_raw = batch['Y_raw'].cpu().numpy()

            pred_p, pred_d = model(x)
            loss = dual_head_loss(pred_p, pred_d, y)

            bs = x.size(0)
            total_loss += loss.item() * bs
            n_samples  += bs

            # Reassemble into original OUTPUT_JOINTS order (proximal + distal)
            pred_flat = torch.cat([pred_p, pred_d], dim=1)   # (B, 24)
            pred_raw  = pred_flat.view(bs, len(OUTPUT_JOINTS), 3).cpu().numpy()
            if y_mean is not None and y_std is not None:
                pred_raw = pred_raw * y_std + y_mean

            preds_list.append(pred_raw)
            trues_list.append(y_raw)

    preds = np.concatenate(preds_list, axis=0)
    trues = np.concatenate(trues_list, axis=0)
    mse   = float(np.mean((preds - trues) ** 2))
    mae   = float(np.mean(np.abs(preds - trues)))
    per_joint_mae = np.mean(np.abs(preds - trues), axis=(0, 2))
    return {
        'loss':          total_loss / max(n_samples, 1),
        'mse':           mse,
        'mae':           mae,
        'per_joint_mae': per_joint_mae,
        'preds':         preds,
        'trues':         trues,
    }

print(f'Helpers defined. DISTAL_LOSS_WEIGHT={DISTAL_LOSS_WEIGHT}')

## 9. LOSO Cross-Validation

**Protocol:**
- Fold *k*: subject *k* → test set
- Remaining subjects → training pool
- `VAL_RATIO` of training sequences held out for early stopping
- Model re-initialised from scratch each fold

In [ ]:
n_subjects     = len(processed_sessions)
fold_results   = []
fold_histories = []

print('=' * 60)
print('CNN-Transformer Dual-Head v3 — LOSO Cross-Validation')
print('Changes vs v3: single FC head → dual heads (proximal + distal)')
print(f'               DISTAL_LOSS_WEIGHT={DISTAL_LOSS_WEIGHT}')
print('=' * 60)

for test_subj_idx in range(n_subjects):

    test_name   = processed_sessions[test_subj_idx]['name']
    train_idxs  = [i for i in range(n_subjects) if i != test_subj_idx]
    train_names = [processed_sessions[i]['name'] for i in train_idxs]

    print(f"\n{'─' * 60}")
    print(f'Fold {test_subj_idx + 1}/{n_subjects}  |  '
          f'Test: [{test_name}]  |  Train: {train_names}')
    print(f"{'─' * 60}")

    # ── Session subsets ──────────────────────────────────────
    train_sessions = [copy.deepcopy(processed_sessions[i]) for i in train_idxs]
    test_sessions  = [copy.deepcopy(processed_sessions[test_subj_idx])]

    train_Y = np.concatenate([s['Y'] for s in train_sessions], axis=0)
    Y_mean  = train_Y.mean(axis=0)
    Y_std   = train_Y.std(axis=0) + 1e-6
    for s in train_sessions:
        s['Y'] = (s['Y'] - Y_mean) / Y_std
    for s in test_sessions:
        s['Y'] = (s['Y'] - Y_mean) / Y_std

    # ── Sliding-window items ─────────────────────────────────
    all_train_items = build_items(train_sessions, SEQ_LEN)
    test_items      = build_items(test_sessions,  SEQ_LEN)

    random.seed(SEED)
    random.shuffle(all_train_items)
    n_val            = max(1, int(len(all_train_items) * VAL_RATIO))
    val_items_fold   = all_train_items[-n_val:]
    train_items_fold = all_train_items[:-n_val]

    print(f'  Train sequences : {len(train_items_fold)}')
    print(f'  Val   sequences : {len(val_items_fold)}')
    print(f'  Test  sequences : {len(test_items)}')

    # ── DataLoaders ──────────────────────────────────────────
    kw = dict(batch_size=BATCH_SIZE, num_workers=0,
              pin_memory=torch.cuda.is_available())
    train_loader = DataLoader(
        SlidingWindowDataset(train_sessions, train_items_fold, SEQ_LEN),
        shuffle=True, **kw)
    val_loader = DataLoader(
        SlidingWindowDataset(train_sessions, val_items_fold, SEQ_LEN),
        shuffle=False, **kw)
    test_loader = DataLoader(
        SlidingWindowDataset(test_sessions, test_items, SEQ_LEN),
        shuffle=False, **kw)

    # ── Fresh model ──────────────────────────────────────────
    torch.manual_seed(SEED)
    model     = make_model()
    optimizer = make_optimizer(model)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=1e-5
    )
    model_path = MODELS_DIR / f'cnn_transformer_dualhead_v3_fold{test_subj_idx+1}_{test_name}.pt'

    # ── Training loop with early stopping ───────────────────
    history       = {'train_loss': [], 'val_loss': [], 'lr': []}
    best_val      = float('inf')
    best_epoch    = -1
    patience_left = PATIENCE

    for epoch in range(1, EPOCHS + 1):
        train_loss = run_epoch(model, train_loader, optimizer=optimizer)
        val_loss   = run_epoch(model, val_loader,   optimizer=None)
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['lr'].append(current_lr)

        show_lr = (epoch == 1 or epoch % 50 == 0)
        print(f'  Epoch {epoch:03d}/{EPOCHS} | '
              f'train={train_loss:.6f} | val={val_loss:.6f}'
              + (f' | lr={current_lr:.2e}' if show_lr else ''), end='')

        if val_loss < best_val:
            best_val      = val_loss
            best_epoch    = epoch
            patience_left = PATIENCE
            torch.save(model.state_dict(), model_path)
            print('  ✓', end='')
        else:
            patience_left -= 1
        print()

        scheduler.step()

        if patience_left <= 0:
            print(f'  Early stopping at epoch {epoch}.')
            break

    print(f'  Best epoch: {best_epoch},  best val loss: {best_val:.6f}')

    # ── Evaluate best checkpoint on test set ─────────────────
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    results = evaluate(model, test_loader, y_mean=Y_mean, y_std=Y_std)

    print(f'\n  ── Fold {test_subj_idx+1} Test Results ──')
    print(f'  Test subject : {test_name}')
    print(f'  Test MSE     : {results["mse"]:.6f}')
    print(f'  Test MAE     : {results["mae"]:.6f}')
    print(f'  Per-joint MAE:')
    for jname, jmae in zip(OUTPUT_JOINTS, results['per_joint_mae']):
        group = 'proximal' if jname in PROXIMAL_JOINTS else 'distal '
        print(f'    [{group}] {jname:<22s}: {jmae:.4f}')

    fold_results.append({
        'fold':       test_subj_idx + 1,
        'test_name':  test_name,
        'best_epoch': best_epoch,
        'best_val':   best_val,
        **{k: results[k] for k in ('loss', 'mse', 'mae', 'per_joint_mae', 'preds', 'trues')},
    })
    fold_histories.append(history)

## 10. Aggregated results

In [ ]:
mses    = [r['mse'] for r in fold_results]
maes    = [r['mae'] for r in fold_results]
pj_maes = np.stack([r['per_joint_mae'] for r in fold_results])

print('=' * 60)
print('CNN-Transformer Dual-Head v3 — Aggregated Results')
print('=' * 60)

print(f"\n{'Fold':<6} {'Subject':<16} {'MSE':>10} {'MAE':>10}")
print('─' * 46)
for r in fold_results:
    print(f"  {r['fold']:<4} {r['test_name']:<16} "
          f"{r['mse']:>10.6f} {r['mae']:>10.6f}")
print('─' * 46)
print(f"  {'Mean':<20} {np.mean(mses):>10.6f} {np.mean(maes):>10.6f}")
print(f"  {'Std':<20} {np.std(mses):>10.6f}  {np.std(maes):>10.6f}")

print('\nPer-joint MAE — mean ± std across folds:')
print('  [Proximal]')
for j, jname in enumerate(PROXIMAL_JOINTS):
    vals = pj_maes[:, j]
    print(f'    {jname:<22s}: {vals.mean():.4f} ± {vals.std():.4f}')
print('  [Distal]')
for j, jname in enumerate(DISTAL_JOINTS):
    idx  = j + len(PROXIMAL_JOINTS)
    vals = pj_maes[:, idx]
    print(f'    {jname:<22s}: {vals.mean():.4f} ± {vals.std():.4f}')

## 11. Loss curves

In [ ]:
fig, axes = plt.subplots(1, len(fold_histories), figsize=(5 * len(fold_histories), 4))
if len(fold_histories) == 1:
    axes = [axes]

for ax, history, r in zip(axes, fold_histories, fold_results):
    ep = range(1, len(history['train_loss']) + 1)
    ax.plot(ep, history['train_loss'], label='train')
    ax.plot(ep, history['val_loss'],   label='val')
    ax.axvline(r['best_epoch'], color='red', linestyle='--', linewidth=0.8,
               label=f"best ep {r['best_epoch']}")
    ax.set_title(f"Fold {r['fold']} — test: {r['test_name']}")
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('CNN-Transformer Dual-Head — Training Curves', fontweight='bold')
plt.tight_layout()
plt.savefig('dualhead_loss_curves.png', dpi=150)
plt.show()

## 12. Per-joint MAE bar chart

近端（proximal）和远端（distal）关节用不同颜色区分，
直观显示双头设计是否有效降低了 heel/toe 的 MAE。

In [ ]:
mean_pj = pj_maes.mean(axis=0)
std_pj  = pj_maes.std(axis=0)
short_names = [j.replace('left-', 'L-').replace('right-', 'R-')
                .replace('-big-toe-tip', 'Toe') for j in OUTPUT_JOINTS]

# Color: blue for proximal, orange for distal
colors = ['steelblue'] * len(PROXIMAL_JOINTS) + ['darkorange'] * len(DISTAL_JOINTS)

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(OUTPUT_JOINTS))
bars = ax.bar(x, mean_pj, yerr=std_pj, capsize=4, color=colors, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(short_names, rotation=30, ha='right')
ax.set_ylabel('MAE (normalised coords)')
ax.set_title('CNN-Transformer Dual-Head — Per-joint MAE (mean ± std over folds)')
ax.grid(True, axis='y', alpha=0.3)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='steelblue',  alpha=0.85, label='Proximal (hip, knee) — head_proximal'),
    Patch(facecolor='darkorange', alpha=0.85, label='Distal (heel, toe)   — head_distal'),
]
ax.legend(handles=legend_elements, fontsize=9)
plt.tight_layout()
plt.savefig('dualhead_per_joint_mae.png', dpi=150)
plt.show()

## 13. Skeleton visualisation and gait animation

In [ ]:
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D

CONN_IDX = [
    (0, 2), (2, 4), (4, 6),
    (1, 3), (3, 5), (5, 7),
    (0, 1),
]


def plot_skeleton_comparison(preds, trues, frame_idx=0, fold_name='subject'):
    fig = plt.figure(figsize=(14, 5))
    titles     = ['Ground Truth', 'Prediction', 'Overlay (GT=blue, Pred=red)']
    data_pairs = [
        (trues[frame_idx], None),
        (preds[frame_idx], None),
        (trues[frame_idx], preds[frame_idx]),
    ]
    for col, (title, (d1, d2)) in enumerate(zip(titles, data_pairs)):
        ax = fig.add_subplot(1, 3, col + 1, projection='3d')

        def draw(pts, color, alpha=1.0):
            ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=color, s=50, alpha=alpha)
            for i, j in CONN_IDX:
                ax.plot([pts[i,0], pts[j,0]], [pts[i,1], pts[j,1]],
                        [pts[i,2], pts[j,2]], c=color, lw=2, alpha=alpha)

        draw(d1, '#2196F3')
        if d2 is not None:
            draw(d2, '#F44336', alpha=0.7)
        ax.set_title(f'{title}\n[{fold_name}] frame {frame_idx}', fontsize=9)
        ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
        ax.view_init(elev=15, azim=-70)
    plt.tight_layout()
    plt.savefig(f'dualhead_skeleton_{fold_name}.png', dpi=150)
    plt.show()


def plot_joint_trajectories(preds, trues, fold_name='subject', n_frames=300):
    preds, trues = preds[:n_frames], trues[:n_frames]
    t = np.arange(len(preds))
    fig, axes = plt.subplots(8, 3, figsize=(15, 20))
    fig.suptitle(f'Joint Trajectories — [{fold_name}]', fontsize=12, fontweight='bold')
    for j, jname in enumerate(OUTPUT_JOINTS):
        short = jname.replace('left-', 'L-').replace('right-', 'R-')
        group = 'P' if jname in PROXIMAL_JOINTS else 'D'
        for ax_i, ax_n in enumerate(['X', 'Y', 'Z']):
            ax = axes[j, ax_i]
            ax.plot(t, trues[:, j, ax_i], 'b-',  lw=1.5, label='GT')
            ax.plot(t, preds[:, j, ax_i], 'r--', lw=1.2, label='Pred')
            if ax_i == 0:
                ax.set_ylabel(f'[{group}] {short}\n{ax_n}', fontsize=7)
            if j == 0 and ax_i == 0:
                ax.legend(fontsize=6)
            ax.tick_params(labelsize=6)
            ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'dualhead_trajectories_{fold_name}.png', dpi=150)
    plt.show()


def plot_error_heatmap(preds, trues, fold_name='subject'):
    per_frame_mae = np.mean(np.abs(preds - trues), axis=2)
    fig, ax = plt.subplots(figsize=(16, 4))
    im = ax.imshow(per_frame_mae.T, aspect='auto', cmap='RdYlGn_r')
    ylabels = [
        f'[P] {j}' if j in PROXIMAL_JOINTS else f'[D] {j}'
        for j in [jn.replace('left-','L-').replace('right-','R-').replace('-big-toe-tip','Toe')
                  for jn in OUTPUT_JOINTS]
    ]
    ax.set_yticks(range(8))
    ax.set_yticklabels(ylabels, fontsize=9)
    ax.set_xlabel('Frame')
    ax.set_title(f'Per-frame Per-joint MAE — [{fold_name}]  [P]=proximal [D]=distal')
    plt.colorbar(im, ax=ax, label='MAE')
    plt.tight_layout()
    plt.savefig(f'dualhead_heatmap_{fold_name}.png', dpi=150)
    plt.show()


for result in fold_results:
    name  = result['test_name']
    preds = result['preds']
    trues = result['trues']
    print(f'\n=== Visualising [{name}] ===')
    plot_skeleton_comparison(preds, trues, frame_idx=50, fold_name=name)
    plot_joint_trajectories(preds, trues, fold_name=name, n_frames=300)
    plot_error_heatmap(preds, trues, fold_name=name)

## 14. Proximal vs Distal MAE comparison

专门比较双头设计对两个关节组的效果。

In [ ]:
proximal_maes = pj_maes[:, :len(PROXIMAL_JOINTS)]   # (folds, 4)
distal_maes   = pj_maes[:, len(PROXIMAL_JOINTS):]   # (folds, 4)

print('=' * 50)
print('Proximal vs Distal MAE Summary')
print('=' * 50)
print(f'\nProximal (hip + knee): {proximal_maes.mean():.4f} ± {proximal_maes.std():.4f}')
print(f'Distal   (heel + toe): {distal_maes.mean():.4f} ± {distal_maes.std():.4f}')
print(f'\nRatio (distal / proximal): {distal_maes.mean() / proximal_maes.mean():.2f}×')
print('(Compare this ratio with v3 results to see if dual-head reduces the gap)')

print('\nLaTeX table:')
print(r'\begin{table}[h]')
print(r'\centering')
print(r'\begin{tabular}{lcc}')
print(r'\toprule')
print(r'Test Subject & MSE & MAE \\')
print(r'\midrule')
for r in fold_results:
    print(f"  {r['test_name']} (Fold {r['fold']}) & "
          f"{r['mse']:.4f} & {r['mae']:.4f} \\\\")
print(r'\midrule')
print(f'  Mean $\\pm$ Std & '
      f'{np.mean(mses):.4f} $\\pm$ {np.std(mses):.4f} & '
      f'{np.mean(maes):.4f} $\\pm$ {np.std(maes):.4f} \\\\')
print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\caption{CNN-Transformer Dual-Head v3 LOSO results.}')
print(r'\label{tab:dualhead_loso}')
print(r'\end{table}')